<a href="https://colab.research.google.com/github/rburchf1/AI102Challenges/blob/main/ryan-burchfield-ai-102-final-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SECTION 1: PROJECT INTRODUCTION

# ORPS Categorization Assistant

## An NLP-Based Chatbot for Evaluating DOE Occurrence Reporting Criteria

**Student:** Ryan Burchfield  
**Course:** AI 102 — Natural Language Processing

## Project Overview

This project develops a retrieval-based natural language processing chatbot that assists users in evaluating whether a workplace accident or operational event may meet the Occurrence Reporting and Processing System criteria contained in DOE O 232.2A, Attachment 2.

The chatbot accepts a natural-language description of an occurrence and compares it with previously categorized occurrence records. It then recommends the most likely reporting criterion or indicates that no reporting criterion can be confidently identified.

The project applies the following NLP concepts:

- Noise reduction and text standardization
- Tokenization and lemmatization
- Part-of-speech tagging
- Dependency parsing
- Negation detection
- Semantic similarity
- TF-IDF vectorization
- Cosine similarity
- Sentence embeddings
- Topic modeling
- Context and missing-information analysis
- Confidence-based response selection

The chatbot is intended as a decision-support prototype. It does not replace review by an authorized occurrence-reporting professional.

## Section 2: Environment Setup

This section installs and imports the Python libraries required for data processing, NLP analysis, machine learning, visualization, and model evaluation.

The primary libraries used are:

- **pandas** and **NumPy** for data management
- **spaCy** for tokenization, lemmatization, part-of-speech tagging, dependency parsing, and named entity recognition
- **scikit-learn** for TF-IDF vectorization, cosine similarity, topic modeling, dataset splitting, and evaluation
- **sentence-transformers** for semantic sentence embeddings
- **Matplotlib** for charts and evaluation figures

In [18]:
# Install the additional libraries required by the notebook.

!pip install -q spacy
!pip install -q sentence-transformers
!pip install -q openpyxl

# Download the small English spaCy language model.
!python -m spacy download en_core_web_sm -q



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 45.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
import spacy
spacy.load('en_core_web_sm')

In [20]:
# Import Python libraries
from pathlib import Path
import re
import string
import warnings

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Natural language processing
import spacy

# Machine-learning utilities
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import NMF

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Google Drive connection
from google.colab import drive

# Suppress unnecessary warning messages
warnings.filterwarnings("ignore")

# Configure pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

print("Python libraries imported successfully.")

Python libraries imported successfully.


In [5]:
# Load spaCy's English language model.
# This model provides tokenization, lemmatization,
# part-of-speech tagging, named entity recognition,
# and dependency parsing.

nlp = spacy.load("en_core_web_sm")

print("spaCy English model loaded successfully.")
print(f"Pipeline components: {nlp.pipe_names}")

spaCy English model loaded successfully.
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## Section 3: Google Drive Connection and Project Paths

The project files are stored in Google Drive so the notebook can access the dataset, save processed data, retain trained models, and export evaluation results across multiple Colab sessions.

The project uses separate folders for:

- Raw source data
- Processed data
- Saved models
- Figures
- Evaluation results
- Chatbot examples
- Report materials

The raw source data will not be modified. All cleaned or transformed files will be saved in the processed-data folder.

In [6]:
# Connect the Colab notebook to Google Drive.

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Define the main project directory.

PROJECT_DIR = Path(
    "/content/drive/MyDrive/AI102_ORPS_Chatbot"
)

# Data folders
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

# Notebook folder
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"

# Model folder
MODEL_DIR = PROJECT_DIR / "models"

# Output folders
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
EVALUATION_DIR = OUTPUT_DIR / "evaluation"
CHATBOT_EXAMPLES_DIR = OUTPUT_DIR / "chatbot_examples"
LOG_DIR = OUTPUT_DIR / "logs"

# Report folders
REPORT_DIR = PROJECT_DIR / "report"
REPORT_FIGURE_DIR = REPORT_DIR / "figures"

# Create folders if any are missing.
project_folders = [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    NOTEBOOK_DIR,
    MODEL_DIR,
    FIGURE_DIR,
    EVALUATION_DIR,
    CHATBOT_EXAMPLES_DIR,
    LOG_DIR,
    REPORT_DIR,
    REPORT_FIGURE_DIR
]

for folder in project_folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders verified successfully.")

Project folders verified successfully.


In [8]:
# Display the directories used by the notebook.

folder_paths = {
    "Project directory": PROJECT_DIR,
    "Raw data": RAW_DATA_DIR,
    "Processed data": PROCESSED_DATA_DIR,
    "Models": MODEL_DIR,
    "Figures": FIGURE_DIR,
    "Evaluation results": EVALUATION_DIR,
    "Chatbot examples": CHATBOT_EXAMPLES_DIR,
    "Report": REPORT_DIR
}

for folder_name, folder_path in folder_paths.items():
    print(f"{folder_name}:")
    print(f"  {folder_path}")

Project directory:
  /content/drive/MyDrive/AI102_ORPS_Chatbot
Raw data:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/data/raw
Processed data:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/data/processed
Models:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/models
Figures:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/outputs/figures
Evaluation results:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/outputs/evaluation
Chatbot examples:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/outputs/chatbot_examples
Report:
  /content/drive/MyDrive/AI102_ORPS_Chatbot/report


In [9]:
# Define the input and output dataset locations.

EVENT_DATA_PATH = RAW_DATA_DIR / "event_data.csv"
CLEAN_DATA_PATH = PROCESSED_DATA_DIR / "event_data_cleaned.csv"

# These files may be added later.
BLIND_TEST_PATH = RAW_DATA_DIR / "blind_test_set.csv"
DOE_CRITERIA_PATH = RAW_DATA_DIR / "doe_criteria.csv"

print("Main event dataset:")
print(EVENT_DATA_PATH)

print("\nFile exists:")
print(EVENT_DATA_PATH.exists())

Main event dataset:
/content/drive/MyDrive/AI102_ORPS_Chatbot/data/raw/event_data.csv

File exists:
True


In [10]:
# Confirm that the source dataset exists before loading it.

if not EVENT_DATA_PATH.exists():
    raise FileNotFoundError(
        "event_data.csv was not found at the expected location:\n"
        f"{EVENT_DATA_PATH}\n\n"
        "Verify the Google Drive folder name and file location."
    )

# Load the source CSV file.
event_df = pd.read_csv(EVENT_DATA_PATH)

print("Dataset loaded successfully.")
print(f"Number of rows: {event_df.shape[0]:,}")
print(f"Number of columns: {event_df.shape[1]}")

display(event_df.head())

Dataset loaded successfully.
Number of rows: 6,734
Number of columns: 2


,Event Description,Categorization
0,"On January 2, 2018, the Environment, Safety, Health and Quality Office received results of initial personal and area cadmium (Cd) exposure monitoring for an activity that was performed on December 13, 2017. The activity, performed under a Safe Work Permit (SWP), involved cleaning out the Vapor T...","2A(7) - Personnel exposure to chemical, biological or physical hazards above limits established in 10 CFR Part 851, Worker Safety and Health Program (see 10 CFR Section 851.23, Safety and Health Standards), but below levels deemed IDLH."
1,"On January 16, 2018, an employee unplugged the power cord of their laptop power supply from the recessed 120 Volt electrical receptacle located in the center of the conference room table. The employee held onto the plug with their left hand as they pulled the plug out of the receptacle, and as t...","2D(1) - Any unexpected or unintended personal contact (e.g., burn, shock, injury, etc.) with a hazardous energy source (e.g., live electrical power circuit, mechanical hazards, steam, pressurized gas, etc.)."
2,"On January 15, 2018, two researches (Researcher #1 and #2) were reconnecting radio frequency (RF) cables in the chasse under the Combi 5 Physical Vapor Deposition (PVD) tool in Lab C112 of NREL's Solar Energy Research Facility (SERF) after completing routine maintenance on the system's nitrogen ...","2D(2) - Any failure to follow a prescribed hazardous energy control process that results in potential worker exposure to uncontrolled hazardous energy (e.g., live electrical power circuit, powered mechanical hazards, steam, pressurized gas, etc.); OR any discovery of an uncontrolled hazardous en..."
3,"Event Sequence and Event Discovery: On Tuesday, April 24, 2018, at approximately 6:30 PM, a post-doctoral researcher (Researcher #1) in the Thin Film Electrochemical Laboratory (Lab E129 in the Solar Energy Research Facility, or SERF) initiated the automated catalyst regeneration sequence on an ...","4B(4) - A facility operational event which resulted in an adverse effect on safety, such as, but not limited to: (a) an inadvertent facility or operations shutdown (i.e., a change of operational mode or curtailment of work or processes); (b) a manual facility or operations shutdown due to alarm ..."
4,"On July 11, 2018, at approximately 9:00 AM, a subcontract worker (Worker #1) was observed standing on the southeast roof extension of the Science and Technology Facility (S&TF) without utilizing appropriate fall protection. Consequently, the worker was exposed to a fall hazard of approximately 1...","10(2) - A near miss to an injury, where something physically happened that was unexpected or unintended AND where no barrier prevented an event from having a reportable consequence (i.e., happenstance was the main reason the event did not result in a reportable injury)."


In [11]:
print("Original column names:")

for column in event_df.columns:
    print(f"- {column}")

print("\nDataset information:")
event_df.info()

Original column names:
- Event Description
- Categorization

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6734 entries, 0 to 6733
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Event Description  6730 non-null   object
 1   Categorization     6734 non-null   object
dtypes: object(2)
memory usage: 105.3+ KB


In [12]:
dataset_summary = pd.DataFrame({
    "Measure": [
        "Number of records",
        "Number of columns",
        "Missing values",
        "Completely duplicated rows"
    ],
    "Value": [
        len(event_df),
        len(event_df.columns),
        int(event_df.isna().sum().sum()),
        int(event_df.duplicated().sum())
    ]
})

display(dataset_summary)

,Measure,Value
0,Number of records,6734
1,Number of columns,2
2,Missing values,4
3,Completely duplicated rows,12


In [13]:
print("Setup verification")
print("-" * 40)

print(f"Google Drive mounted: {PROJECT_DIR.exists()}")
print(f"event_data.csv located: {EVENT_DATA_PATH.exists()}")
print(f"Dataset loaded: {len(event_df) > 0}")
print(f"spaCy loaded: {nlp is not None}")

print("\nDataset columns:")
print(event_df.columns.tolist())

Setup verification
----------------------------------------
Google Drive mounted: True
event_data.csv located: True
Dataset loaded: True
spaCy loaded: True

Dataset columns:
['Event Description', 'Categorization']


## Section 4: Dataset Validation and Cleaning

This section validates the event dataset and prepares it for NLP analysis.

The cleaning process includes:

- Standardizing column names
- Identifying the occurrence-description and categorization columns
- Removing rows with missing or blank values
- Standardizing non-reportable labels
- Identifying exact duplicates
- Identifying descriptions assigned to multiple criteria
- Preserving the original text before preprocessing
- Saving a cleaned working dataset

The original raw dataset is not modified.

In [15]:
# Create a working copy so the raw DataFrame remains unchanged.

df = event_df.copy()

print("Working copy created.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Working copy created.
Rows: 6,734
Columns: 2


In [23]:
description_blank = (
    df["event_description"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

categorization_blank = (
    df["categorization"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

missing_summary = pd.DataFrame({
    "Field": [
        "event_description",
        "categorization"
    ],
    "Missing_or_blank_records": [
        int(description_blank.sum()),
        int(categorization_blank.sum())
    ]
})

display(missing_summary)

,Field,Missing_or_blank_records
0,event_description,4
1,categorization,0


In [24]:
# Remove unusable records

rows_before_removal = len(df)

df = df.loc[
    ~description_blank & ~categorization_blank
].copy()

rows_removed = rows_before_removal - len(df)

print(f"Rows before removal: {rows_before_removal:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Rows remaining: {len(df):,}")

Rows before removal: 6,734
Rows removed: 4
Rows remaining: 6,730


In [25]:
# Convert both fields to strings and remove unnecessary outer whitespace.

df["event_description"] = (
    df["event_description"]
    .astype(str)
    .str.strip()
)

df["categorization"] = (
    df["categorization"]
    .astype(str)
    .str.strip()
)

print("Text fields converted to strings and trimmed.")

Text fields converted to strings and trimmed.


In [26]:
def standardize_category_label(label):
    """
    Standardize criterion labels while preserving criterion codes.
    """
    label = str(label).strip().upper()

    # Replace repeated whitespace with a single space.
    label = re.sub(r"\s+", " ", label)

    non_reportable_variations = {
        "NON REPORTABLE",
        "NON-REPORTABLE",
        "NONREPORTABLE",
        "NOT REPORTABLE",
        "NO CRITERIA MET",
        "NO CRITERION MET",
        "NO ORPS CRITERIA MET",
        "NONE"
    }

    if label in non_reportable_variations:
        return "NON_REPORTABLE"

    return label


df["categorization"] = df["categorization"].apply(
    standardize_category_label
)

print("Categorization labels standardized.")

Categorization labels standardized.


In [27]:
# Review categorization labels

category_counts = (
    df["categorization"]
    .value_counts()
    .rename_axis("categorization")
    .reset_index(name="record_count")
)

category_counts["percentage"] = (
    category_counts["record_count"] / len(df) * 100
).round(2)

print(f"Unique category labels: {df['categorization'].nunique()}")

display(category_counts)

Unique category labels: 179


,categorization,record_count,percentage
0,"2D(2) - ANY FAILURE TO FOLLOW A PRESCRIBED HAZARDOUS ENERGY CONTROL PROCESS THAT RESULTS IN POTENTIAL WORKER EXPOSURE TO UNCONTROLLED HAZARDOUS ENERGY (E.G., LIVE ELECTRICAL POWER CIRCUIT, POWERED MECHANICAL HAZARDS, STEAM, PRESSURIZED GAS, ETC.); OR ANY DISCOVERY OF AN UNCONTROLLED HAZARDOUS EN...",1159,17.22
1,"4A(1) - PERFORMANCE DEGRADATION OF ANY SAFETY CLASS (SC) OR SAFETY SIGNIFICANT (SS) STRUCTURE, SYSTEM, OR COMPONENT (SSC), OR ANY SUPPORT SYSTEM THAT IS REQUIRED FOR SAFETY OPERATION OF THE SC OR SS SSCS, WHICH PREVENTS SATISFACTORY PERFORMANCE OF ITS DESIGN FUNCTION WHEN IT IS REQUIRED TO BE OP...",1095,16.27
2,"2A(5) - ANY SINGLE OCCURRENCE RESULTING IN AN OCCUPATIONAL INJURY OR EXPOSURE THAT: (A) REQUIRES IN PATIENT HOSPITALIZATION FOR MORE THAN 48 HOURS, COMMENCING WITHIN SEVEN DAYS FROM THE DATE THE INJURY OR EXPOSURE WAS RECEIVED; (B) RESULTS IN A FRACTURE OF ANY BONE (EXCEPT BONE CHIPS; SIMPLE FRA...",929,13.80
3,"10(2) - A NEAR MISS TO AN INJURY, WHERE SOMETHING PHYSICALLY HAPPENED THAT WAS UNEXPECTED OR UNINTENDED AND WHERE NO BARRIER PREVENTED AN EVENT FROM HAVING A REPORTABLE CONSEQUENCE (I.E., HAPPENSTANCE WAS THE MAIN REASON THE EVENT DID NOT RESULT IN A REPORTABLE INJURY).",519,7.71
4,3B(2) - DETERMINATION OF A POSITIVE UNREVIEWED SAFETY QUESTION (USQ) THAT REVEALS A CURRENTLY EXISTING INADEQUACY IN THE DOCUMENTED SAFETY ANALYSIS.,409,6.08
...,...,...,...
174,"10(3) - ANY OCCURRENCE THAT MAY RESULT IN A SIGNIFICANT CONCERN BY AFFECTED STATE, TRIBAL, OR LOCAL OFFICIALS, PRESS, OR GENERAL POPULATION; THAT COULD DAMAGE THE CREDIBILITY OF THE DEPARTMENT; OR THAT MAY RESULT IN INQUIRIES TO HEADQUARTERS. 5B(1) - ANY OCCURRENCE INCLUDING RELEASES CAUSING SIG...",1,0.01
175,"2A(3) - ANY SINGLE OCCURRENCE, INJURY, OR EXPOSURE RESULTING IN AN OCCUPATIONAL INJURY THAT REQUIRES IN-PATIENT HOSPITALIZATION FOR FIVE OR MORE DAYS, COMMENCING WITHIN SEVEN DAYS FROM THE DATE THE INJURY. 10(1) - AN EVENT, CONDITION, OR SERIES OF EVENTS THAT DOES NOT MEET ANY OF THE OTHER REPOR...",1,0.01
176,"2B(2) - ANY FIRE THAT: (A) ACTIVATES A FIXED AUTOMATIC FIRE SUPPRESSION SYSTEM (E.G., CLEAN AGENT OR WET PIPE AUTOMATIC SPRINKLER PROTECTION), (B) TAKES LONGER THAN TEN MINUTES TO EXTINGUISH FOLLOWING THE INITIATION OF FIREFIGHTING EFFORTS BY THE EMERGENCY RESPONSE ORGANIZATION, OR (C) DISRUPTS ...",1,0.01
177,"2C(1) - ANY UNPLANNED EXPLOSION THAT DISRUPTS NORMAL OPERATIONS. 10(2) - A NEAR MISS TO AN INJURY, WHERE SOMETHING PHYSICALLY HAPPENED THAT WAS UNEXPECTED OR UNINTENDED AND WHERE NO BARRIER PREVENTED AN EVENT FROM HAVING A REPORTABLE CONSEQUENCE (I.E., HAPPENSTANCE WAS THE MAIN REASON THE EVENT ...",1,0.01
